In [ ]:
from transformers import AutoVideoProcessor, AutoModel, AutoModelForVideoClassification
import torch
import numpy as np
from torchcodec.decoders import VideoDecoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Option A: Raw feature extraction ---
hf_repo = "facebook/vjepa2-vitl-fpc64-256"
processor = AutoVideoProcessor.from_pretrained(hf_repo)
model = AutoModel.from_pretrained(hf_repo).to(device)

# Load the downloaded video
vr = VideoDecoder("sample_video.mp4")
frame_idx = np.linspace(0, len(vr) - 1, 64, dtype=int)  # sample 64 frames
video = vr.get_frames_at(indices=frame_idx).data  # T x C x H x W

inputs = processor(video, return_tensors="pt").to(device)
with torch.no_grad():
    video_embeddings = model.get_vision_features(**inputs)

print(f"Video embedding shape: {video_embeddings.shape}")

# --- Option B: Video classification (with SSv2 action head) ---
hf_repo_cls = "facebook/vjepa2-vitl-fpc64-256-ssv2"
model_cls = AutoModelForVideoClassification.from_pretrained(hf_repo_cls).to(device)
processor_cls = AutoVideoProcessor.from_pretrained(hf_repo_cls)

inputs_cls = processor_cls(video, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model_cls(**inputs_cls)
logits = outputs.logits

print("\nTop 5 predicted actions:")
top5 = logits.topk(5).indices[0]
for idx in top5:
    print(f"  Class {idx.item()}: {model_cls.config.id2label[idx.item()]}")

Using device: cpu


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Video embedding shape: torch.Size([1, 8192, 1024])


OSError: facebook/vjepa2-vitl-fpc64-256-ssv2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
%%bash
pip install -q transformers torch torchcodec numpy requests

In [ ]:
import requests

# Download a sample short video to use for testing
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/classroom.mp4"
video_path = "sample_video.mp4"

print("Downloading sample video...")
response = requests.get(video_url, stream=True)
with open(video_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
print("Download complete!")

Download complete!


In [ ]:
from transformers import AutoVideoProcessor, AutoModel, AutoModelForVideoClassification
import torch
import numpy as np
from torchcodec.decoders import VideoDecoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Option A: Raw feature extraction ---
hf_repo = "facebook/vjepa2-vitl-fpc64-256"
processor = AutoVideoProcessor.from_pretrained(hf_repo)
model = AutoModel.from_pretrained(hf_repo).to(device)

# Load video using VideoDecoder and get the sampled frames correctly
vr = VideoDecoder("sample_video.mp4")
frame_idx = np.linspace(0, len(vr) - 1, 64, dtype=int)  # sample 64 frames
video = vr.get_frames_at(indices=frame_idx).data  # T x C x H x W

inputs = processor(video, return_tensors="pt").to(device)
with torch.no_grad():
    video_embeddings = model.get_vision_features(**inputs)

print(f"Video embedding shape: {video_embeddings.shape}")

# --- Option B: Video classification ---
hf_repo_cls = "facebook/vjepa2-vitl-fpc64-256-ssv2"
model_cls = AutoModelForVideoClassification.from_pretrained(hf_repo_cls).to(device)
processor_cls = AutoVideoProcessor.from_pretrained(hf_repo_cls)

inputs_cls = processor_cls(video, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model_cls(**inputs_cls)
logits = outputs.logits

print("\nTop 5 predicted actions:")
top5 = logits.topk(5).indices[0]
for idx in top5:
    print(f"  Class {idx.item()}: {model_cls.config.id2label[idx.item()]}")

Using device: cpu


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Video embedding shape: torch.Size([1, 8192, 1024])


OSError: facebook/vjepa2-vitl-fpc64-256-ssv2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`